## Setup

This notebook uses Python. Run cells in order from top to bottom. Ensure you've put an OpenAI API Key in .env. 

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from tqdm.auto import tqdm
tqdm.pandas()

## Configuration

Edit the values in the cell below to match your data **before running the notebook**.

In [2]:
# Path to data
INPUT_PATH = "data/nicar-2026-schedule.csv"

# --- Required: at least one text column ---
TEXT_COL_TITLE   = "title"   # Title of text (in this case, we're just using the full text -- no title)
TEXT_COL_CONTENT = "description"     # Content field (No content column for plain text files)

# --- Output ---
OUTPUT_DIR = "output"   # Folder to save results (created automatically if it doesn't exist)

## Load Data

In [3]:
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df):,} rows from CSV")

df.head()

Loaded 232 rows from CSV


,session_id,date,start_time,end_time,title,description,recorded,type,skill_level,speakers,tracks,room,room_os,level,cost
0,1311,2026-03-05,8:00 AM,8:45 AM,Welcome to NICAR! First-timers welcome and net...,Welcome to the conference! Hear from IRE staff...,False,Networking,Beginner,"Diana Fuentes, Investigative Reporters & Edito...",Event,White River E,NaN,1,NaN
1,1002,2026-03-05,9:00 AM,10:00 AM,How to get data out of complex documents using AI,Learn how newsrooms get important information ...,True,Demo,Beginner,"Duy Nguyen, The New York Times; Disha Raychaud...","AI, Tools & Tech",White River D,NaN,1,NaN
2,1069,2026-03-05,9:00 AM,10:00 AM,Using data to investigate the global surveilla...,"Around the world, shadowy firms are siphoning ...",True,Panel,Beginner,"Andrew Couts, WIRED; Gabriel Geiger, Lighthous...","Data analysis, Data viz, Beat reporting",White River E,NaN,1,NaN
3,1160,2026-03-05,9:00 AM,10:00 AM,State secrets: A workflow for using state-leve...,Learn how to find and use personal financial a...,False,Demo,Intermediate,"Audrey Nielsen, MuckRock; Diara Townes, NC Local","Local, Public records, Research, Elections",102,BYO,1,NaN
4,1193,2026-03-05,9:00 AM,10:00 AM,Data journalism for new reporters,"""If you build it - they will come"": Many local...",True,Panel,Beginner,"Helena Bengtsson, Gota Media / Bonnier News Lo...",Local,White River G,NaN,1,NaN


## Embeddings

Generate a vector embedding for each row.

In [4]:
import tiktoken

ENCODING   = "cl100k_base"
MAX_TOKENS = 8000
enc        = tiktoken.get_encoding(ENCODING)

def make_combined_text(row):
    """Combine title and (optional) content text columns for embedding."""
    title = str(row[TEXT_COL_TITLE]) if pd.notna(row[TEXT_COL_TITLE]) else ""
    title = "Title: {title}" if title else ""

    content = str(row[TEXT_COL_CONTENT]) if pd.notna(row[TEXT_COL_CONTENT]) else ""
    content = f"Content: {content}" if content else ""

    return title + content

df["combined"] = df.apply(make_combined_text, axis=1)
df["n_tokens"] = df["combined"].apply(lambda x: len(enc.encode(x)))
df = df.sort_values("n_tokens", ascending=False)
print(f"Longest article: {df['n_tokens'].max():,} tokens | Average: {df['n_tokens'].mean():.0f} tokens")
df[["combined", "n_tokens"]].head()

Longest article: 254 tokens | Average: 107 tokens


,combined,n_tokens
62,Title: {title}Content: Polars is a modern data...,254
83,Title: {title}Content: Being a news manager is...,252
146,Title: {title}Content: We've all been there re...,248
155,"Title: {title}Content: You've got a leak, a na...",242
130,Title: {title}Content: Writing Python scripts ...,229


# Filter out any text that will not fit
Using estimated tokens, filter out any large text rows.
For other projects, you may need to cut off large text or deal with them usig other methods. 

In [5]:
too_long = df[df["n_tokens"] > MAX_TOKENS]
print(f"Removing {len(too_long):,} articles exceeding {MAX_TOKENS:,} tokens.")
df = df[df["n_tokens"] <= MAX_TOKENS].reset_index(drop=True)
print(f"{len(df):,} rows remaining.")

Removing 0 articles exceeding 8,000 tokens.
232 rows remaining.


### OpenAI API

Embeddings from OpenAI. Requires an API key in `.env` and the `openai` + `python-dotenv` packages.

In [6]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

EMBEDDING_MODEL = "text-embedding-3-small"

def get_embeddings(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

def process_in_batches(df, column, batch_size=30):
    all_embeddings = []
    texts = df[column].tolist()
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        all_embeddings.extend(get_embeddings(batch))
    return all_embeddings

df["embedding"] = process_in_batches(df, "combined")

df = df.drop(columns=["combined", "n_tokens"])
df.head()

  0%|          | 0/8 [00:00<?, ?it/s]

,session_id,date,start_time,end_time,title,description,recorded,type,skill_level,speakers,tracks,room,room_os,level,cost,embedding
0,1061,2026-03-05,5:00 PM,6:00 PM,Analyzing election data with Polars and Python,Polars is a modern dataframe library for Pytho...,False,Hands-on,Intermediate,"Oliver Hawkins, Financial Times; Eade Hemingwa...",Data analysis,104,OSX,1,NaN,"[-0.041999876499176025, 0.021270208060741425, ..."
1,1276,2026-03-06,9:00 AM,11:15 AM,Managing investigators: Leading those born to ...,"Being a news manager is already tough, but wha...",False,Master Class,Intermediate,"Emma Carew Grovum, Kimbap Media; Josh Hinkle, ...",Management,209,BYO,2,0.0,"[0.006235484499484301, 0.04406556487083435, 0...."
2,1123,2026-03-07,9:00 AM,10:00 AM,"How ""Skills"" solve the biggest pitfall in usin...",We've all been there recently. Jazzed at exper...,False,Hands-on,Advanced,"Aaron Kessler, Associated Press",AI,104,OSX,1,NaN,"[0.010714713484048843, 0.0720641016960144, 0.0..."
3,1287,2026-03-07,9:00 AM,10:00 AM,Find public records and leaks: The new OCCRP A...,"You've got a leak, a name, or a suspicious com...",False,Hands-on,Beginner,"Ezana Ćeman, Organized Crime and Corruption Re...",International,201,BYO,2,NaN,"[0.01572592183947563, 0.041844889521598816, 0...."
4,1106,2026-03-06,3:45 PM,4:45 PM,Modern document processing with Natural PDF,Writing Python scripts to extract data from PD...,False,Hands-on,Advanced,"Jonathan Soma, Columbia University","AI, Tools & Tech",104,OSX,1,NaN,"[-0.015374742448329926, 0.0468563586473465, 0...."


## Dimensionality Reduction (t-SNE)

Embeddings are high-dimensional vectors (~1,536 dimensions). t-SNE projects them to 2D (x, y) for plotting. This runs every time the notebook is executed.

In [7]:
from sklearn.manifold import TSNE

print("Running t-SNE (this may take a minute)...")
matrix = np.array(df["embedding"].tolist())
tsne = TSNE(
    n_components=2,
    perplexity=min(30, len(df) // 5),  # scales with dataset size
    random_state=42,
    init="pca",           # stable, deterministic
    learning_rate="auto") # adapts to dataset size
vis_dims = tsne.fit_transform(matrix)
df["x"] = vis_dims[:, 0]
df["y"] = vis_dims[:, 1]
print("Done.")

Running t-SNE (this may take a minute)...
Done.


## Topic Modeling (HDBSCAN)

[HDBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html) clusters points in 2D space based on density — sessions in the same cluster tend to be about the same topic. Unlike DBSCAN, it automatically finds clusters of varying density without requiring an `eps` parameter.

- **Noise points** (`-1`) = sessions that don't fit into any cluster, labeled "Uncategorized"
- Adjust `min_cluster_size` (minimum sessions per cluster) and `min_samples` (controls how conservative clustering is) if you get too few or too many clusters.

In [8]:
from sklearn.cluster import HDBSCAN

coords_2d = df[["x", "y"]].values

db = HDBSCAN(min_cluster_size=3, min_samples=2)
cluster_labels = db.fit_predict(coords_2d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = (cluster_labels == -1).sum()
print(f"Clusters found: {n_clusters}")
print(f"Noise points:   {n_noise}")
print(f"Cluster sizes:  {sorted([(l, (cluster_labels==l).sum()) for l in set(cluster_labels) if l != -1], key=lambda x: -x[1])}")

df["cluster"] = cluster_labels

Clusters found: 35
Noise points:   46
Cluster sizes:  [(np.int64(8), np.int64(18)), (np.int64(9), np.int64(12)), (np.int64(27), np.int64(10)), (np.int64(34), np.int64(8)), (np.int64(12), np.int64(7)), (np.int64(24), np.int64(7)), (np.int64(28), np.int64(7)), (np.int64(17), np.int64(6)), (np.int64(19), np.int64(6)), (np.int64(31), np.int64(6)), (np.int64(0), np.int64(5)), (np.int64(11), np.int64(5)), (np.int64(13), np.int64(5)), (np.int64(14), np.int64(5)), (np.int64(21), np.int64(5)), (np.int64(25), np.int64(5)), (np.int64(26), np.int64(5)), (np.int64(29), np.int64(5)), (np.int64(1), np.int64(4)), (np.int64(4), np.int64(4)), (np.int64(6), np.int64(4)), (np.int64(7), np.int64(4)), (np.int64(10), np.int64(4)), (np.int64(22), np.int64(4)), (np.int64(23), np.int64(4)), (np.int64(30), np.int64(4)), (np.int64(2), np.int64(3)), (np.int64(3), np.int64(3)), (np.int64(5), np.int64(3)), (np.int64(15), np.int64(3)), (np.int64(16), np.int64(3)), (np.int64(18), np.int64(3)), (np.int64(20), np.int64(

## Name Clusters with LLM

Send all cluster contents to the LLM in one prompt and get back whimsical fantasy-map region names.

In [9]:
import json
from collections import defaultdict

# Group session titles by cluster (skip noise)
cluster_titles = defaultdict(list)
for _, row in df.iterrows():
    label = row["cluster"]
    if label != -1:
        cluster_titles[int(label)].append(row[TEXT_COL_TITLE])

# Build a single prompt with all clusters
cluster_block = "\n\n".join(
    f"Cluster {cid}:\n" + "\n".join(f"  - {t}" for t in titles)
    for cid, titles in sorted(cluster_titles.items())
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a cartographer naming regions on a fantasy map of a data journalism conference. "
                "For each cluster, invent a whimsical geographic place name that captures what makes it "
                "DISTINCT from the other clusters — not just what the sessions are about, but what sets "
                "this group apart. Names should be 2-4 words using geographic suffixes like: "
                "Bay, Cape, Cove, Delta, Flats, Forest, Gorge, Gulf, Harbor, Heights, Hills, Isle, "
                "Junction, Lagoon, Land, Mesa, Moors, Mountains, Peninsula, Plateau, Prairie, Range, "
                "Rapids, Reach, Ridge, Shore, Sound, Strait, Summit, Vale, Valley, Wastes, Wilds. "
                "Every name must be unique. "
                "Respond with ONLY valid JSON: {\"0\": \"Name Here\", \"1\": \"Name Here\", ...}"
            ),
        },
        {
            "role": "user",
            "content": f"Name each of these clusters with a unique map region name:\n\n{cluster_block}",
        },
    ],
    temperature=0.7,
    max_tokens=4000,
)

raw = response.choices[0].message.content.strip()

# Strip markdown fences if present
if raw.startswith("```"):
    raw = "\n".join(raw.split("\n")[1:])
    raw = raw.rsplit("```", 1)[0].strip()

cluster_names = {int(k): v for k, v in json.loads(raw).items()}

for cluster_id, name in sorted(cluster_names.items()):
    print(f"Cluster {cluster_id:2d} ({len(cluster_titles[cluster_id]):2d} sessions): {name}")

# Attach names to dataframe
df["cluster_name"] = df["cluster"].apply(
    lambda c: cluster_names.get(int(c), "Uncategorized") if c != -1 else "Uncategorized"
)

Cluster  0 ( 5 sessions): Tomorrow's News Cove
Cluster  1 ( 4 sessions): Document Delta
Cluster  2 ( 3 sessions): Lunch Truck Lagoon
Cluster  3 ( 3 sessions): Newcomers' Junction
Cluster  4 ( 4 sessions): Evening Enchantment Harbor
Cluster  5 ( 3 sessions): International Insight Isle
Cluster  6 ( 4 sessions): Mentor Meadow
Cluster  7 ( 4 sessions): Networking Heights
Cluster  8 (18 sessions): Spreadsheet Summit
Cluster  9 (12 sessions): Ridge of R
Cluster 10 ( 4 sessions): Spatial Analysis Plateau
Cluster 11 ( 5 sessions): Collaboration Coast
Cluster 12 ( 7 sessions): Geospatial Wilds
Cluster 13 ( 5 sessions): Mapping Moors
Cluster 14 ( 5 sessions): Visualization Vale
Cluster 15 ( 3 sessions): Design Dunes
Cluster 16 ( 3 sessions): Automation Abyss
Cluster 17 ( 6 sessions): Sports Data Shore
Cluster 18 ( 3 sessions): Scraping Straits
Cluster 19 ( 6 sessions): Python Peninsula
Cluster 20 ( 3 sessions): FOIA Falls
Cluster 21 ( 5 sessions): Immigration Isle
Cluster 22 ( 4 sessions): Workf

## Export

In [10]:
# --- Full dataset (all columns, including embedding vectors) ---
full_path = os.path.join(OUTPUT_DIR, "data-with-embeddings.csv")
df.to_csv(full_path, index=False)
print(f"Full dataset    → '{full_path}'")

# --- Visualization dataset ---
# Renames columns to match the visualization tool's expected schema:
#   title, text, date, org, x, y, url
rename_map = {}
if TEXT_COL_TITLE:   rename_map[TEXT_COL_TITLE]   = "title"
if TEXT_COL_CONTENT: rename_map[TEXT_COL_CONTENT] = "text"

vis_df = df.rename(columns=rename_map)

vis_cols = [c for c in ["title", "text", "date", "org", "x", "y", "url", "cluster_name"]
            if c in vis_df.columns]

vis_path = os.path.join(OUTPUT_DIR, "data-for-visualization.csv")
vis_df[vis_cols].to_csv(vis_path, index=False)
print(f"Visualization   → '{vis_path}'")
print(f"\nColumns exported: {vis_cols}")

Full dataset    → 'output/data-with-embeddings.csv'
Visualization   → 'output/data-for-visualization.csv'

Columns exported: ['title', 'text', 'date', 'x', 'y', 'cluster_name']


## Interactive Chart

In [11]:
import plotly.express as px
import plotly.graph_objects as go
import textwrap

def wrap_hover(text, width=60):
    if pd.isna(text):
        return ""
    lines = textwrap.wrap(str(text), width=width)
    return "<br>".join(lines)

plot_df = df.copy()
plot_df["_title"] = plot_df[TEXT_COL_TITLE].apply(lambda t: wrap_hover(t, width=60))

hover_name_col = "filename" if "filename" in df.columns else TEXT_COL_TITLE
hover_data = {"_title": True, "cluster_name": True, "x": False, "y": False}
if TEXT_COL_CONTENT and TEXT_COL_CONTENT in df.columns:
    plot_df["_content"] = plot_df[TEXT_COL_CONTENT].apply(lambda t: wrap_hover(t, width=60))
    hover_data["_content"] = True

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="cluster_name",
    hover_name=hover_name_col,
    hover_data=hover_data,
    title="Semantic Map — NICAR 2026",
    width=1000,
    height=750,
)
fig.update_traces(marker=dict(size=9, opacity=0.7))

# Add cluster name labels at the centroid of each cluster
labeled = plot_df[plot_df["cluster"] != -1]
centroids = labeled.groupby("cluster_name")[["x", "y"]].mean().reset_index()
for _, row in centroids.iterrows():
    fig.add_annotation(
        x=row["x"], y=row["y"],
        text=f"<b>{row['cluster_name']}</b>",
        showarrow=False,
        font=dict(size=10, color="black"),
        bgcolor="rgba(255,255,255,0.6)",
        borderpad=2,
    )

fig.update_layout(
    legend_title_text="Region",
    hoverlabel=dict(namelength=-1),
)
fig.show()